# point_probe_offcentre:点提示头用偏心点重训

**背景**:点提示头(输入一个 patch 的 DINOv2 特征和射线,从冻结的 3DRAE token 里读出 3D 位置)训练时只见过物体中心的投影点。dino.txt 选出的 patch 落在物体上任意位置,这时它的误差从 0.27 m 涨到 0.47 m,大物体更明显。

**做法**:重训点提示头(`jepa3d/geo_probe/train.py --box-prompts SHARE`),训练时每个物体以 SHARE 的概率把提示点从中心换成「射线穿过该物体 3D 框的随机 patch」里的随机像素。其余设置与原来相同(958 个训练场景,100 epoch,3DRAE 冻结)。训完用 `improve_teacher_grounding.py` 在同样的验证样本上测。

- `centre`:原来的点提示头(只用中心点训练)
- `box05`:SHARE = 0.5
- `box10`:SHARE = 1.0

注意:框内的 patch 可能被前面的东西挡住,此时 patch 特征是遮挡物的。训练和测试都有这种噪声。

In [1]:
import json
import os
import numpy as np
import pandas as pd
pd.set_option("display.width", 220)

def load(path):
    return json.load(open(path)) if os.path.exists(path) else None

def pct(m, key):
    return f'{100 * m[key]["value"]:.1f}' if key in m else ""

def pct_ci(m, key):
    return f'{100 * m[key]["value"]:.1f} [{100 * m[key]["ci95"][0]:.1f}, {100 * m[key]["ci95"][1]:.1f}]' if key in m else ""

def metres(m, key, ci=False):
    if key not in m:
        return ""
    v = m[key]
    return f'{v["value"]:.2f} [{v["ci95"][0]:.2f}, {v["ci95"][1]:.2f}]' if ci else f'{v["value"]:.2f}'

runs = {"centre": load("../../teacher_hit_rate/results/improve.json"), "box05": load("../results/box05.json"),
        "box10": load("../results/box10.json")}
runs = {k: v["metrics"] for k, v in runs.items() if v}
ROWS = [("gt", "物体中心的真实投影"), ("box", "框内随机 patch"), ("224/centered/argmax", "dino.txt:224 / centered / argmax(原做法)"),
        ("ms/softmax/argmax", "dino.txt:ms / softmax / argmax"), ("ms/softmax/consensus+", "dino.txt:ms / softmax / consensus+")]

def table(key, fmt, **kw):
    return pd.DataFrame([{"提示点来源": label, **{name: fmt(m, f"{v}/{key}", **kw) for name, m in runs.items()}} for v, label in ROWS])

## 1. 位置误差中位数(m),可见物体

In [2]:
table("visible/position_m", metres, ci=True)

,提示点来源,centre,box05,box10
0,物体中心的真实投影,"0.27 [0.25, 0.29]","0.29 [0.27, 0.31]","0.31 [0.29, 0.33]"
1,框内随机 patch,"0.47 [0.45, 0.49]","0.43 [0.40, 0.45]","0.43 [0.41, 0.45]"
2,dino.txt:224 / centered / argmax(原做法),"0.67 [0.63, 0.73]","0.57 [0.53, 0.62]","0.56 [0.52, 0.61]"
3,dino.txt:ms / softmax / argmax,"0.65 [0.60, 0.70]","0.55 [0.52, 0.59]","0.53 [0.50, 0.58]"
4,dino.txt:ms / softmax / consensus+,"0.63 [0.59, 0.68]","0.53 [0.49, 0.57]","0.52 [0.48, 0.56]"


## 2. 象限准确率(%),可见物体

In [3]:
table("visible/quadrant_acc", pct_ci)

,提示点来源,centre,box05,box10
0,物体中心的真实投影,"84.3 [82.9, 85.7]","83.2 [81.8, 84.5]","82.4 [81.0, 83.9]"
1,框内随机 patch,"74.4 [73.2, 75.7]","75.6 [74.5, 76.9]","75.5 [74.3, 76.8]"
2,dino.txt:224 / centered / argmax(原做法),"50.6 [48.9, 52.6]","51.3 [49.6, 53.4]","51.7 [49.9, 53.8]"
3,dino.txt:ms / softmax / argmax,"51.9 [50.0, 53.7]","52.8 [50.8, 54.7]","53.0 [51.1, 55.0]"
4,dino.txt:ms / softmax / consensus+,"51.5 [49.7, 53.3]","52.6 [50.6, 54.4]","53.0 [51.1, 55.1]"


## 3. 按物体大小:框内随机 patch 的位置误差(m),中心在视野内的物体

In [4]:
pd.DataFrame([{"最长边": label, **{name: metres(m, f"box/in_view_size_{g}/position_m") for name, m in runs.items()}}
              for g, label in (("lt0.5m", "< 0.5 m"), ("0.5-1m", "0.5–1 m"), ("gt1m", "> 1 m"))])

,最长边,centre,box05,box10
0,< 0.5 m,0.30,0.30,0.31
1,0.5–1 m,0.41,0.38,0.39
2,> 1 m,0.62,0.56,0.54


## 4. dino.txt 指中 / 没指中时的位置误差(m),原做法选点

从逐物体记录里算,中心在视野内的物体。

In [5]:
def hit_miss(path, variant="224/centered/argmax"):
    rows = [json.loads(line) for line in open(path)]
    rows = [r[variant] for r in rows if r["in_view"] and variant in r]
    hit = np.array([r[1] for r in rows]); err = np.array([r[0] for r in rows])
    return {"指中率": f"{100 * hit.mean():.1f}%", "指中时 (m)": f"{np.median(err[hit]):.2f}", "没指中时 (m)": f"{np.median(err[~hit]):.2f}"}

paths = {"centre": "../../teacher_hit_rate/results/improve.objects.jsonl", "box05": "../results/box05.objects.jsonl",
         "box10": "../results/box10.objects.jsonl"}
pd.DataFrame({k: hit_miss(p) for k, p in paths.items() if os.path.exists(p)}).T

,指中率,指中时 (m),没指中时 (m)
centre,62.2%,0.41,1.78
box05,62.2%,0.36,1.74
box10,62.2%,0.35,1.76


## 结论

- **偏心点重训有效**:dino.txt 选点时的位置误差从 0.67 m 降到 0.57 m(box05)/ 0.56 m(box10);框内随机 patch 从 0.47 m 降到 0.43 m,大物体(> 1 m)从 0.62 m 降到 0.54–0.56 m。
- **代价很小**:用真实中心做提示时 0.27 m → 0.29 m(box05)/ 0.31 m(box10)。
- **两个比例差别在置信区间内**。box05 对中心点损失更小,后续用 box05。
- **象限准确率几乎不变**(50.6% → 51–52%):没指中时误差仍是 1.7–1.8 m,重训只改善了指中时的误差(0.41 → 0.36 m)。方向类指标受指中率限制,不受点提示头限制。
- 加上高分辨率和 softmax 打分后最好是 0.52–0.53 m / 53%。

**局限**:验证集 55 个 ScanNet 场景;点提示头没有在 ScanNet++ 上训练或验证过。